In [2]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings


In [3]:

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7706.54it/s]


In [4]:
vector_DB_load_from_local = FAISS.load_local(
    "vector_db",
    embedding_model,
    allow_dangerous_deserialization=True
)
print(vector_DB_load_from_local)

In [5]:
user_query = "What were the main causes for increased lead times mentioned in the report?"

In [18]:
relevant_chunk_documents = vector_DB_load_from_local.similarity_search(
    user_query,
    k=3
)

relevant_chunk_documents

[Document(id='3852c890-ce5e-4d2e-9e5a-31384c03fe75', metadata={'page': 2, 'chunk_id': 1}, page_content='Operational Observations (cid:127) Region A experienced a 12% increase in expedited shipments due to stockouts. (cid:127) Region B showed a 7-day increase in average lead time during monsoon months. (cid:127) Spares accuracy (picks vs orders) remained at 98.2% but returns increased slightly due to labeling errors. Notes on data sources: Data aggregated from ERP, WMS, and Vendor portals. Timestamp alignment used UTC.'),
 Document(id='62fad8a3-ecfc-41a5-9a4d-2e4b19066301', metadata={'page': 3, 'chunk_id': 3}, page_content='0 1 2 3 Metric Region A Region B Region C Avg Lead Time (days) 14 21 12 On-time Delivery (%) 93.4 88.1 95.0 Expedited Shipments (%) 6.1 12.5 4.0 Inventory Turns 3.5 2.8 4.1'),
 Document(id='a5f4fd34-69e3-4892-a0bf-0a2779d77400', metadata={'page': 4, 'chunk_id': 4}, page_content='Sample Parts List Part ID Description Category Qty On Hand Lead Time (days) P-1001 Avioni

In [7]:
context = "\n\n".join(
    doc.page_content
    for doc in relevant_chunk_documents
)
context

'Operational Observations (cid:127) Region A experienced a 12% increase in expedited shipments due to stockouts. (cid:127) Region B showed a 7-day increase in average lead time during monsoon months. (cid:127) Spares accuracy (picks vs orders) remained at 98.2% but returns increased slightly due to labeling errors. Notes on data sources: Data aggregated from ERP, WMS, and Vendor portals. Timestamp alignment used UTC.\n\n0 1 2 3 Metric Region A Region B Region C Avg Lead Time (days) 14 21 12 On-time Delivery (%) 93.4 88.1 95.0 Expedited Shipments (%) 6.1 12.5 4.0 Inventory Turns 3.5 2.8 4.1\n\nSample Parts List Part ID Description Category Qty On Hand Lead Time (days) P-1001 Avionics Control Module Control 25 30 P-1002 Navigation Sensor Sensor 10 45 P-1003 Hydraulic Actuator Actuator 5 60 P-1004 Power Regulator Power 40 20 P-1005 Comms Antenna Antenna 18 25 Table notes: Lead time is the supplier-provided estimate; Qty On Hand is snapshot at close of quarter.'

In [8]:
prompt = f"""
Your task is to answer the user's question ONLY using the information provided in the Context.

STRICT RULES:

1. Use ONLY the Context.
2. Never use outside knowledge.
3. If answer is missing, reply exactly:
Information not available in the document.
4. Answer in bullet points.
5. Keep each bullet short.

Context:
{context}

Question:
{user_query}

Answer:
"""

In [9]:
load_dotenv()


True

In [31]:
print("create llm")
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.getenv("GROQ_API_KEY"),
    
    temperature=0.3
)



create llm


In [32]:
#LLM generate answer

result = llm.invoke(prompt) # prompt + original user query + user_query_embedding

print(result.content)

* Stockouts in Region A 
* Monsoon months in Region B


In [39]:
import json

with open("queries_groq.json", "r", encoding="utf-8") as f:
    queries = json.load(f)

print(queries)

[{'query': 'What were the main causes for increased lead times mentioned in the report?'}, {'query': 'List the KPIs for Region B from the Q3 table.'}, {'query': 'Which parts have lead time more than 30 days?'}, {'query': 'What is the procedure for critical parts exceptions and the SLA for documenting root cause?'}]


In [40]:
def retrieve(query, top_k=3):

    docs = vector_DB_load_from_local.similarity_search_with_score(
        query,
        k=top_k
    )

    return docs

In [46]:
def answer_query(query):

    retrieved_docs = retrieve(query)

    context = ""

    retrieved_chunks = []

    for doc, score in retrieved_docs:

        context += doc.page_content + "\n\n"

        retrieved_chunks.append(
            {
                "page": doc.metadata.get("page", "Unknown"),
                "score": float(score)
            }
        )

    prompt = f"""
You are a question answering assistant.

Answer ONLY from the provided context.

If the answer is not present in the context, reply exactly:

I don't know from the document.

Context:
{context}

Question:
{query}

Answer:
"""

    result = llm.invoke(prompt)

    answer = result.content.strip()

    return {
        "query": query,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }

In [47]:
with open("queries.json", "r", encoding="utf-8") as f:

    queries = json.load(f)

print(queries) 

[{'query': 'What were the main causes for increased lead times mentioned in the report?'}, {'query': 'List the KPIs for Region B from the Q3 table.'}, {'query': 'Which parts have lead time more than 30 days?'}, {'query': 'What is the procedure for critical parts exceptions and the SLA for documenting root cause?'}]


In [48]:
results = []

for item in queries:

    q = item["query"]

    print("Processing:", q)

    output = answer_query(q)

    results.append(output)

Processing: What were the main causes for increased lead times mentioned in the report?
Processing: List the KPIs for Region B from the Q3 table.
Processing: Which parts have lead time more than 30 days?
Processing: What is the procedure for critical parts exceptions and the SLA for documenting root cause?


In [49]:
import json

with open("answers_groq.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("answers_groq.json created successfully.")

answers_groq.json created successfully.
